# Aim
The aim of this notebook is to test the analysis pipeline for Thomas's data on a subset of images and compare to manual quantification.



# 1. Functions and imports

In [6]:
from imaris_ims_file_reader.ims import ims
import numpy as np
import os
import re
from tqdm import tqdm
import pyclesperanto_prototype as cle
import matplotlib.pyplot as plt
import napari
from skimage.morphology import skeletonize, medial_axis
from skan.csr import skeleton_to_csgraph, Skeleton, summarize
from skan import draw
from skimage.feature import shape_index
from skimage.measure import regionprops
from scipy.spatial import cKDTree
import pandas as pd
from stackview import insight
from apoc import ObjectSegmenter
import seaborn as sns
from scipy.ndimage import distance_transform_edt, center_of_mass
from skimage.segmentation import find_boundaries
from skimage.filters import meijering


In [7]:
import limoncello as lc

In [8]:
import contextlib
from functools import wraps

def suppress_print_keep_tqdm(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        with open(os.devnull, 'w') as f, contextlib.redirect_stdout(f):
            return func(*args, **kwargs)
    return wrapper

In [9]:

ims_path = "H:/tlobri/2. Experiments, Differentiations and Adaptations/4. Cilia on Neurites/EXP_E6_250807/250819 - BC43 IF WTSII ROIs/EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(9).ims"

print(cle.available_device_names(dev_type="gpu"))
cle.select_device('NVIDIA GeForce RTX 4090')  # optional but good practice
print(cle.get_device())

['NVIDIA GeForce RTX 4090', 'gfx1036']
<NVIDIA GeForce RTX 4090 on Platform: NVIDIA CUDA (1 refs)>


In [20]:
import numpy as np
import pandas as pd
from scipy.ndimage import map_coordinates

def assign_label_features(
    cilia_labels: np.ndarray,
    cilia_centroids: np.ndarray,
    cilia_ids: np.ndarray,
    dist_to_neurite: np.ndarray,
    nearest_skel_idx: np.ndarray,
    distance_map_neurites: np.ndarray,
    distance_map_nuclei: np.ndarray,
    map_ratio: np.ndarray,
    max_cilia_dist_cutoff_um: float,
    file: str,
    object_type: str = "cilia"
) -> pd.DataFrame:
    """
    Assign features to cilia objects based on centroids and distance maps.

    Parameters
    ----------
    cilia_centroids : np.ndarray
        Array of shape (n_objects, 3) with (z, y, x) coordinates.
    cilia_ids : np.ndarray
        Array of cilia labels corresponding to centroids.
    dist_to_neurite : np.ndarray
        Distance map to neurites (3D volume).
    nearest_skel_idx : np.ndarray
        Nearest skeleton indices array of shape (3, Z, Y, X).
    distance_map_neurites : np.ndarray
        Distance transform of neurites.
    distance_map_nuclei : np.ndarray
        Distance transform of nuclei.
    map_ratio : np.ndarray
        Ratio map.
    max_cilia_dist_cutoff_um : float
        Maximum distance allowed to consider assignment.
    file : str
        Filename for this set of cilia (used in DataFrame).
    cilia_labels_shape : tuple
        Shape of the cilia label volume (for clipping coordinates).
    object_type : str, optional
        Type of object, by default "cilia".

    Returns
    -------
    pd.DataFrame
        DataFrame with one row per cilium that passed the distance threshold.
    """
    rows = []

    for cid, centroid in zip(cilia_ids, cilia_centroids):
        z, y, x = np.round(centroid).astype(int)

        # Clip to volume bounds
        z = np.clip(z, 0, cilia_labels.shape[0] - 1)
        y = np.clip(y, 0, cilia_labels.shape[1] - 1)
        x = np.clip(x, 0, cilia_labels.shape[2] - 1)

        # Distance to neurite at centroid
        d = map_coordinates(dist_to_neurite, np.array(centroid).reshape(3, 1), order=1)[0]

        if d > max_cilia_dist_cutoff_um:
            continue

        # Nearest skeleton voxel
        sz, sy, sx = nearest_skel_idx[:, z, y, x]

        dt_neurite = distance_map_neurites[sz, sy, sx]
        dt_nuclei = distance_map_nuclei[sz, sy, sx]
        ratio = map_ratio[sz, sy, sx]

        rows.append(
            {
                "filename": file,
                "cilia_id": cid,
                "coords": [centroid[0],centroid[1],centroid[2]], # z,y,x
                "distance_to_neurite_um": d,
                "ratio": ratio,
                "log_ratio": np.log(ratio),
                "dt_neurite": dt_neurite,
                "dt_nuclei": dt_nuclei,
                "log_dt_neurite": np.log(dt_neurite),
                "log_dt_nuclei": np.log(dt_nuclei),
                "object_type": object_type,
            }
        )

    df = pd.DataFrame(rows)

    if df.empty:
        print(f"No valid {object_type} found in {file}")

    return df

In [11]:
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

def pair_cilia_basal_bidirectional(
    cilia_df, basal_df, threshold_um=20.0,
    cilia_coord_cols=("centroid_z","centroid_y","centroid_x"),
    basal_coord_cols=("centroid_z","centroid_y","centroid_x")
):
    cilia_coords = cilia_df[list(cilia_coord_cols)].to_numpy()
    basal_coords = basal_df[list(basal_coord_cols)].to_numpy()

    n_cilia = len(cilia_coords)
    n_basal = len(basal_coords)

    # Initialize output columns
    cilia_out = cilia_df.copy()
    basal_out = basal_df.copy()
    for df, obj_type in [(cilia_out, "cilia"), (basal_out, "basal_body")]:
        df["paired_id"] = np.nan
        df["distance_um"] = np.nan
        df["pairing_status"] = "lonely"
        df["object_type"] = obj_type

    if n_cilia == 0 or n_basal == 0:
        return pd.concat([cilia_out, basal_out], ignore_index=True)

    # Compute distances
    dist_matrix = np.linalg.norm(cilia_coords[:, None, :] - basal_coords[None, :, :], axis=2)
    dist_mask = dist_matrix.copy()
    dist_mask[dist_mask > threshold_um] = np.inf

    # Skip Hungarian if no valid pairs
    if np.all(np.isinf(dist_mask)):
        return pd.concat([cilia_out, basal_out], ignore_index=True)

    # One-to-one assignment
    c_idx, b_idx = linear_sum_assignment(dist_mask)

    basal_ids = basal_df["cilia_id"].to_numpy() if "cilia_id" in basal_df.columns else np.arange(n_basal)

    for c, b in zip(c_idx, b_idx):
        if dist_mask[c, b] != np.inf:
            cilia_out.loc[c, "paired_id"] = basal_ids[b]
            cilia_out.loc[c, "distance_um"] = dist_mask[c, b]
            cilia_out.loc[c, "pairing_status"] = "paired"

            basal_out.loc[b, "paired_id"] = cilia_df.loc[c, "cilia_id"] if "cilia_id" in cilia_df.columns else c
            basal_out.loc[b, "distance_um"] = dist_mask[c, b]
            basal_out.loc[b, "pairing_status"] = "paired"

    return pd.concat([cilia_out, basal_out], ignore_index=True)

In [ ]:
def run_pipeline3(
    input_path,
    output_path,
    max_cilia_dist_cutoff_um=2.0,
    max_basal_body_cutoff_um=2.0,
    nuclei_spot_sigma=15,
    tophat_radius=12,
    neurite_spot_sigma=5,
    classifier_path =  r'C:\Users\qfavey\Documents\Code-Thomas-Test\LimonCELLo\segmenters\cilia-segmenter.cl',
):
    os.makedirs(output_path, exist_ok=True)
    csv_dir = os.path.join(output_path, "csv")
    os.makedirs(csv_dir, exist_ok=True)

    all_dfs = []

    for file in tqdm(os.listdir(input_path)):
        if not file.endswith(".ims"):
            continue

        print(f"Processing {file}...")

        # Load data
        a, meta = lc.load_image(os.path.join(input_path, file))
        voxel_size = meta["voxel_size"]
        
        a_norm = lc.normalize_intensity(a,p_low=2,p_high=98)
        
        # Segmentation
        cilia_labels = lc.segment_cilia_ml(
            a[0, 0],
            classifier_path = classifier_path,
            )

        nuclei_labels_otsu = lc.segment_nuclei(
            a_norm[0, 3],
            tophat_radius=(tophat_radius, tophat_radius, tophat_radius),
            spot_sigma=nuclei_spot_sigma,
            outline_sigma=3,
        )

        skeleton, neurites_label = lc.segment_neurites(
            a_norm[0, 1],
            spot_sigma=neurite_spot_sigma,
        )

        basal_bodies_labels = lc.segment_basal_bodies(
            a[0,2]
        )

        # get neurites masks
        neurite_mask = np.asarray(neurites_label) > 0
        skeleton_mask = np.asarray(skeleton) > 0

        # Nuclei distance map
        distance_map_nuclei = distance_transform_edt(
            ~(nuclei_labels_otsu > 0), sampling=voxel_size
        )

        # Neurite thickness proxy (radius)
        distance_map_neurites = distance_transform_edt(
            neurite_mask, sampling=voxel_size
        )

        # Ratio
        map_ratio = np.where(distance_map_neurites > 0,
                     distance_map_nuclei / distance_map_neurites,
                     0)  # or np.nan
        map_ratio[~np.isfinite(map_ratio)] = np.nan

        # Distance to neurites 
        dist_to_neurite = distance_transform_edt(
            ~neurite_mask,
            sampling=voxel_size
        )

        # Nearest SKELETON mapping 
        _, nearest_skel_idx = distance_transform_edt(
            ~skeleton_mask,
            return_indices=True,
            sampling=voxel_size,
        )

        # Cilia centroids
        cilia_ids = np.unique(cilia_labels)
        cilia_ids = cilia_ids[cilia_ids != 0]

        cilia_centroids = center_of_mass(
            cilia_labels > 0, labels=cilia_labels, index=cilia_ids
        )

        # Basal bodies centroids
        basal_bodies_ids = np.unique(basal_bodies_labels)
        basal_bodies_ids = basal_bodies_ids[basal_bodies_ids != 0] # remove background
        basal_bodies_centroids = center_of_mass(
            basal_bodies_labels > 0, labels = basal_bodies_labels, index=basal_bodies_ids
        )


        df_cilia = assign_label_features(
            cilia_labels,
            cilia_centroids,
            cilia_ids,
            dist_to_neurite,
            nearest_skel_idx,
            distance_map_neurites,
            distance_map_nuclei,
            map_ratio,
            max_cilia_dist_cutoff_um,
            file,
            )
        
        if df_cilia.empty:
            print(f"No valid cilia found in {file}")
            continue

        

        df_basal_bodies = assign_label_features(
            basal_bodies_labels,
            basal_bodies_centroids,
            basal_bodies_ids,
            dist_to_neurite,
            nearest_skel_idx,
            distance_map_neurites,
            distance_map_nuclei,
            map_ratio,
            max_basal_body_cutoff_um,
            file,
            "basal_body"
        )

        #paired_df = pair_cilia_basal_bidirectional(df_cilia,df_basal_bodies)
        all_dfs.append(df_basal_bodies)
        all_dfs.append(df_cilia)

        # Overlay visualization (MIP)
        overlay_dir = os.path.join(output_path, "figures", "overlays")
        os.makedirs(overlay_dir, exist_ok=True)

        # MIP of neurite + cilia channels
        neurite_mip = np.max(a_norm[0, 1], axis=0)   # (Y, X)
        cilia_mip = np.max(a_norm[0, 0], axis=0)     # (Y, X)
        nuclei_mip = np.max(a_norm[0,3], axis = 0)
       
        fig, axes = plt.subplots(2,2, figsize=(14,10))

        # Base layer: neurites
        axes[0,0].imshow(neurite_mip, cmap="gray")
        axes[0,0].set_title('Neurites MIP')
        axes[1,1].imshow(cilia_mip,cmap='gray')
        axes[1,1].set_title('Cilia MIP')
        axes[0,1].imshow( np.log(map_ratio[map_ratio.shape[0] // 2]),cmap='coolwarm')
        axes[0,1].set_title('log(ratio) overlay')
        axes[1,0].imshow(nuclei_mip,cmap='gray')
        axes[1,0].set_title('Nuclei MIP')

        # Normalize scores for coloring (robust)
        scores = df_cilia["log_ratio"].values
        valid_scores = scores[np.isfinite(scores)]

        if len(valid_scores) > 0:
            vmin, vmax = np.percentile(valid_scores, [5, 95])
        else:
            vmin, vmax = 0, 1

        norm = plt.Normalize(vmin=vmin, vmax=vmax)
        cmap = plt.cm.coolwarm

        # Overlay cilia centroids
        coords = np.vstack(df_cilia["coords"].values)  # shape (n, 3)

        
        ys = coords[:, 1]
        xs = coords[:, 2]
        colors = cmap(norm(scores))

        axes[0,0].scatter(
            xs,
            ys,
            c=colors,
            s=10,
            edgecolor="black",
            linewidth=0.3,
            
        )
        axes[1,1].scatter(
            xs,
            ys,
            c=colors,
            s=10,
            edgecolor="black",
            linewidth=0.3,
            
        )
        axes[0,1].scatter(
            xs,
            ys,
            c=colors,
            s=10,
            edgecolor="black",
            linewidth=0.3,
            
        )
        axes[1,0].scatter(
            xs,
            ys,
            c=colors,
            s=10,
            edgecolor="black",
            linewidth=0.3,
            
        )

        # Colorbar
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        plt.colorbar(sm, ax=axes[1,0], label="log_ratio")
        plt.colorbar(sm, ax=axes[1,1], label="log_ratio")
        plt.colorbar(sm, ax=axes[0,0], label="log_ratio")
        plt.colorbar(sm, ax=axes[0,1], label="log_ratio")

        plt.axis("off")

        save_path = os.path.join(
            overlay_dir,
            f"{os.path.splitext(file)[0]}_overlay.png"
        )

        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close()

    if len(all_dfs) == 0:
        print("No data processed.")
        return

    final_df = pd.concat(all_dfs, ignore_index=True)

    # Short names
    unique_files = final_df["filename"].unique()
    file_map = {f: f"S{i+1}" for i, f in enumerate(unique_files)}
    final_df["file_short"] = final_df["filename"].map(file_map)

    
    # Classification
    def classify(score):
        if score > 2.5:
            return "axon"
        elif score < 1:
            return "soma"
        else:
            return "ambiguous"

    final_df["class"] = final_df["log_ratio"].apply(classify)

    # Figures (clean + meaningful)
    fig_dir = os.path.join(output_path, "figures")
    os.makedirs(fig_dir, exist_ok=True)

    
    # 1. Multi-panel (ratio + log-ratio)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Ratio
    sns.histplot(final_df["ratio"], kde=True, ax=axes[0, 0])
    axes[0, 0].set_title("Raw Ratio Distribution")

    # Log ratio
    sns.histplot(final_df["log_ratio"], kde=True, ax=axes[0, 1])
    axes[0, 1].set_title("Log Ratio Distribution")

    # Log ratio per sample (boxplot)
    sns.boxplot(data=final_df, x="file_short", y="log_ratio", ax=axes[1, 0])
    axes[1, 0].set_title("Log Ratio per Sample")
    axes[1, 0].tick_params(axis='x', rotation=45)

    # Class distribution per sample
    sns.countplot(data=final_df, x="file_short", hue="class", ax=axes[1, 1])
    axes[1, 1].set_title("Class Distribution per Sample")
    axes[1, 1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "overview_panels.png"), dpi=300)
    plt.close()

    # 2. KDE per sample
    plt.figure(figsize=(10, 6))

    sns.kdeplot(
        data=final_df,
        x="log_ratio",
        hue="file_short",
        common_norm=False,
    )

    plt.title("Log Ratio Distribution per Sample")
    plt.xlabel("Log Ratio")

    plt.savefig(os.path.join(fig_dir, "logratio_kde_per_sample.png"), dpi=300)
    plt.close()

    # QC + Excel
    excel_path = os.path.join(csv_dir, "all_cilia_features.xlsx")

    qc_sample = final_df.groupby("file_short").agg(
        n_cilia=("cilia_id", "count"),
        #n_basal_bodies=("basal_body_id", "count"),
        mean_ratio=("ratio", "mean"),
        std_ratio=("ratio", "std"),
        mean_log_ratio=("log_ratio", "mean"),
        std_log_ratio=("log_ratio", "std"),
        mean_distance=("distance_to_neurite_um", "mean"),
        max_distance=("distance_to_neurite_um", "max"),
    ).reset_index()

    qc_global = pd.DataFrame({
        "metric": [
            "n_total_cilia",
            "mean_ratio",
            "std_ratio",
            "mean_log_ratio",
            "std_log_ratio"
        ],
        "value": [
            len(final_df),
            final_df["ratio"].mean(),
            final_df["ratio"].std(),
            final_df["log_ratio"].mean(),
            final_df["log_ratio"].std(),
        ]
    })

    with pd.ExcelWriter(excel_path) as writer:
        final_df.to_excel(writer, sheet_name="all_data", index=False)
        qc_sample.to_excel(writer, sheet_name="qc_per_sample", index=False)
        qc_global.to_excel(writer, sheet_name="qc_global", index=False)

    print(f"Saved QC Excel: {excel_path}")


    # Scatter plot (the moment of truth)
    plt.figure(figsize=(10, 6))

    sns.scatterplot(
        x='dt_neurite',
        y='dt_nuclei',
        hue='file_short',
        data=final_df,
        palette='tab10'
    )

    plt.xlabel("Assigned Neurite Voxel thickness (um)")
    plt.ylabel("Assigned Neurite Voxel Distance to nuclei (um)")
    plt.title("DT Comparison per Sample")
    plt.legend(bbox_to_anchor=(0.5, 1), loc='upper left')

    fig_dir = os.path.join(output_path, "figures")
    os.makedirs(fig_dir, exist_ok=True)

    plt.savefig(os.path.join(fig_dir, "dt_scatterplot.png"), dpi=300)
    plt.close()

    # Scatter plot (the moment of truth)
    plt.figure(figsize=(10, 6))

    sns.scatterplot(
        x='log_dt_neurite',
        y='log_dt_nuclei',
        hue='file_short',
        data=final_df,
        palette='tab10'
    )

    plt.xlabel("Log of Assigned Neurite Voxel thickness (um)")
    plt.ylabel("Log of Assigned Neurite Voxel Distance to nuclei (um)")
    plt.title("Log DT Comparison per Sample")
    plt.legend(bbox_to_anchor=(0.5, 1), loc='upper left')

    fig_dir = os.path.join(output_path, "figures")
    os.makedirs(fig_dir, exist_ok=True)

    plt.savefig(os.path.join(fig_dir, "log_dt_scatterplot.png"), dpi=300)
    plt.close()


    print(f"Figures saved in: {fig_dir}")
    print("Pipeline complete.")

In [22]:
input_path = r"C:\Users\qfavey\Documents\Thomas-test-data\Data"
output_path="C:/Users/qfavey/Documents/Thomas-test-data/Test_Run"
run_pipeline3(input_path=input_path,output_path=output_path)

  0%|          | 0/3 [00:00<?, ?it/s]

Processing EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(6).ims...
Opening .ims file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(6).ims
Opening readonly file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(6).ims 

Shape (T, C, Z, Y, X): (1, 4, 23, 2040, 2040)
Number of timepoints: 1
Number of channels: 4
Image dimensions (Z, Y, X): (23, 2040, 2040)
Z Resolution: 0.383
XY Resolution: (0.152, 0.152)


C:\Users\qfavey\AppData\Local\Temp\ipykernel_88452\3302895996.py:66: RuntimeWarning: divide by zero encountered in divide
  distance_map_nuclei / distance_map_neurites,
C:\Users\qfavey\AppData\Local\Temp\ipykernel_88452\3302895996.py:66: RuntimeWarning: invalid value encountered in divide
  distance_map_nuclei / distance_map_neurites,
C:\Users\qfavey\AppData\Local\Temp\ipykernel_88452\44327007.py:81: RuntimeWarning: divide by zero encountered in log
  "log_ratio": np.log(ratio),
C:\Users\qfavey\AppData\Local\Temp\ipykernel_88452\44327007.py:85: RuntimeWarning: divide by zero encountered in log
  "log_dt_nuclei": np.log(dt_nuclei),
C:\Users\qfavey\AppData\Local\Temp\ipykernel_88452\3302895996.py:152: RuntimeWarning: divide by zero encountered in log
  axes[0,1].imshow( np.log(map_ratio[map_ratio.shape[0] // 2]),cmap='coolwarm')
 33%|███▎      | 1/3 [00:55<01:50, 55.10s/it]

Processing EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(7).ims...
Opening .ims file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(7).ims
Opening readonly file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(7).ims 

Shape (T, C, Z, Y, X): (1, 4, 23, 2040, 2040)
Number of timepoints: 1
Number of channels: 4
Image dimensions (Z, Y, X): (23, 2040, 2040)
Z Resolution: 0.383
XY Resolution: (0.152, 0.152)
Closing file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(6).ims 



C:\Users\qfavey\AppData\Local\Temp\ipykernel_88452\3302895996.py:66: RuntimeWarning: divide by zero encountered in divide
  distance_map_nuclei / distance_map_neurites,
C:\Users\qfavey\AppData\Local\Temp\ipykernel_88452\3302895996.py:66: RuntimeWarning: invalid value encountered in divide
  distance_map_nuclei / distance_map_neurites,
C:\Users\qfavey\AppData\Local\Temp\ipykernel_88452\44327007.py:81: RuntimeWarning: divide by zero encountered in log
  "log_ratio": np.log(ratio),
C:\Users\qfavey\AppData\Local\Temp\ipykernel_88452\44327007.py:85: RuntimeWarning: divide by zero encountered in log
  "log_dt_nuclei": np.log(dt_nuclei),
C:\Users\qfavey\AppData\Local\Temp\ipykernel_88452\3302895996.py:152: RuntimeWarning: divide by zero encountered in log
  axes[0,1].imshow( np.log(map_ratio[map_ratio.shape[0] // 2]),cmap='coolwarm')
 67%|██████▋   | 2/3 [01:45<00:52, 52.63s/it]

Processing EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(9).ims...
Opening .ims file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(9).ims
Opening readonly file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(9).ims 

Shape (T, C, Z, Y, X): (1, 4, 23, 2040, 2040)
Number of timepoints: 1
Number of channels: 4
Image dimensions (Z, Y, X): (23, 2040, 2040)
Z Resolution: 0.383
XY Resolution: (0.152, 0.152)
Closing file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(7).ims 



C:\Users\qfavey\AppData\Local\Temp\ipykernel_88452\3302895996.py:66: RuntimeWarning: divide by zero encountered in divide
  distance_map_nuclei / distance_map_neurites,
C:\Users\qfavey\AppData\Local\Temp\ipykernel_88452\3302895996.py:66: RuntimeWarning: invalid value encountered in divide
  distance_map_nuclei / distance_map_neurites,
C:\Users\qfavey\AppData\Local\Temp\ipykernel_88452\44327007.py:81: RuntimeWarning: divide by zero encountered in log
  "log_ratio": np.log(ratio),
C:\Users\qfavey\AppData\Local\Temp\ipykernel_88452\44327007.py:85: RuntimeWarning: divide by zero encountered in log
  "log_dt_nuclei": np.log(dt_nuclei),
C:\Users\qfavey\AppData\Local\Temp\ipykernel_88452\3302895996.py:152: RuntimeWarning: divide by zero encountered in log
  axes[0,1].imshow( np.log(map_ratio[map_ratio.shape[0] // 2]),cmap='coolwarm')
100%|██████████| 3/3 [02:38<00:00, 52.73s/it]
c:\Users\qfavey\.conda\envs\napari-env-q\lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid valu

Saved QC Excel: C:/Users/qfavey/Documents/Thomas-test-data/Test_Run\csv\all_cilia_features.xlsx
Figures saved in: C:/Users/qfavey/Documents/Thomas-test-data/Test_Run\figures
Pipeline complete.
Closing file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(9).ims 



In [ ]:
import napari

def run_pipeline2_debug_napari(
    input_path,
    output_path,
    dist_cutoff_um=2.0,
    nuclei_spot_sigma=15,
    tophat_radius=12,
    neurite_spot_sigma=5,
):
    os.makedirs(output_path, exist_ok=True)

    for file in os.listdir(input_path):
        if not file.endswith(".ims"):
            continue

        print(f"Processing {file}...")

               # -------------------------
        # Load data
        # -------------------------
        a, meta = reader.load_image(os.path.join(input_path, file))
        voxel_size = meta["voxel_size"]
        
        a_norm = preprocessing.normalize_intensity(a)
        
        # -------------------------
        # Segmentation
        # -------------------------
        cilia_labels = cilia_process_3d_ml(a[0, 0])

        nuclei_labels = nuclei_process_3d(
            a_norm[0, 3],
            tophat_radius=(tophat_radius, tophat_radius, tophat_radius),
            spot_sigma=nuclei_spot_sigma,
            outline_sigma=3,
        )

        skeleton, neurites_label = neurites_process_gpu(
            a_norm[0, 1],
            nuclei_labels,
            spot_sigma=neurite_spot_sigma,
        )

        neurite_mask = np.asarray(neurites_label) > 0
        skeleton_mask = np.asarray(skeleton) > 0

        # -------------------------
        # Distance maps
        # -------------------------
        distance_map_nuclei = distance_transform_edt(
            ~(nuclei_labels > 0), sampling=voxel_size
        )

        distance_map_neurites = distance_transform_edt(
            neurite_mask, sampling=voxel_size
        )

        dist_to_neurite, nearest_idx = distance_transform_edt(
            ~neurite_mask,
            return_indices=True,
            sampling=voxel_size,
        )

        _, nearest_skel_idx = distance_transform_edt(
            ~skeleton_mask,
            return_indices=True,
            sampling=voxel_size,
        )

        # -------------------------
        # Cilia centroids
        # -------------------------
        cilia_ids = np.unique(cilia_labels)
        cilia_ids = cilia_ids[cilia_ids != 0]

        centroids = center_of_mass(
            cilia_labels > 0, labels=cilia_labels, index=cilia_ids
        )

        centroids = np.array(centroids)

        # -------------------------
        # Napari visualization
        # -------------------------
        viewer = napari.Viewer()

        # Raw channels
        viewer.add_image(a[0, 0], name="Cilia channel", colormap="green")
        viewer.add_image(a[0, 1], name="Neurite channel", colormap="magenta")
        viewer.add_image(a[0, 3], name="Nuclei channel", colormap="blue")

        # Segmentations
        viewer.add_labels(cilia_labels, name="Cilia labels")
        viewer.add_labels(nuclei_labels, name="Nuclei labels")
        viewer.add_labels(neurites_label, name="Neurite mask")

        # Skeleton
        viewer.add_labels(skeleton_mask.astype(int), name="Skeleton")

        # Distance maps
        viewer.add_image(distance_map_nuclei, name="DT nuclei", colormap="inferno")
        viewer.add_image(distance_map_neurites, name="DT neurite (thickness)", colormap="viridis")

        # Distance to neurite
        viewer.add_image(dist_to_neurite, name="Distance to neurite", colormap="magma")

        # Centroids
        if len(centroids) > 0:
            viewer.add_points(
                centroids,
                name="Cilia centroids",
                size=5,
                face_color="red",
            )
        # -------------------------
        # Vectors: cilia → skeleton
        # -------------------------
        vectors = []

        for centroid in centroids:
            z, y, x = np.round(centroid).astype(int)

            z = np.clip(z, 0, skeleton_mask.shape[0] - 1)
            y = np.clip(y, 0, skeleton_mask.shape[1] - 1)
            x = np.clip(x, 0, skeleton_mask.shape[2] - 1)

            # nearest skeleton voxel
            sz, sy, sx = nearest_skel_idx[:, z, y, x]

            origin = np.array([centroid[0], centroid[1], centroid[2]])
            target = np.array([sz, sy, sx])

            direction = target - origin

            vectors.append([origin, direction])

        if len(vectors) > 0:
            vectors = np.array(vectors)

            viewer.add_vectors(
                vectors,
                name="Cilia → Skeleton vectors",
                edge_color="yellow",
                edge_width=1,
                length=1,  # keep real scale
            )

        # -------------------------
        # Ratio overlay (cilia / neurite)
        # -------------------------
        # avoid division by zero
        
        ratio_cilia_overlay = distance_map_nuclei / distance_map_neurites

        viewer.add_image(
            ratio_cilia_overlay,
            name="Cilia / Neurite ratio",
            colormap="magma",
            blending="additive",
        )

        print("Napari viewer opened. Inspect your life choices.")
        napari.run()

In [ ]:
run_pipeline2_debug_napari(input_path=input_path,output_path=output_path)

Processing EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(6).ims...
Opening .ims file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(6).ims
Opening readonly file: C:\Users\qfavey\Documents\Thomas-test-data\Data\EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647_40x_2025-08-19_(6).ims 

Shape (T, C, Z, Y, X): (1, 4, 23, 2040, 2040)
Number of timepoints: 1
Number of channels: 4
Image dimensions (Z, Y, X): (23, 2040, 2040)
Z Resolution: 0.383
XY Resolution: (0.152, 0.152)


TypeError: only length-1 arrays can be converted to Python scalars